In [94]:
import pandas as pd 
import numpy as np

### Exercise Dataset

In [95]:
exercise_df = pd.read_csv("datasets/exercise.csv")

exercise_df.head()

,Unnamed: 0,Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,NaN
1,1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
2,2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
3,3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
4,4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,NaN,NaN


In [96]:
exercise_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2918 entries, 0 to 2917
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  2918 non-null   int64  
 1   Title       2918 non-null   object 
 2   Desc        1368 non-null   object 
 3   Type        2918 non-null   object 
 4   BodyPart    2918 non-null   object 
 5   Equipment   2886 non-null   object 
 6   Level       2918 non-null   object 
 7   Rating      1031 non-null   float64
 8   RatingDesc  862 non-null    object 
dtypes: float64(1), int64(1), object(7)
memory usage: 205.3+ KB


In [97]:
exercise_df.columns

Index(['Unnamed: 0', 'Title', 'Desc', 'Type', 'BodyPart', 'Equipment', 'Level',
       'Rating', 'RatingDesc'],
      dtype='object')

In [98]:
exercise_df.isnull().sum()

Unnamed: 0       0
Title            0
Desc          1550
Type             0
BodyPart         0
Equipment       32
Level            0
Rating        1887
RatingDesc    2056
dtype: int64

### Data Cleaning

In [99]:
exercise_df = exercise_df[
    ["Title", "Type", "BodyPart", "Equipment", "Level"]
]

In [100]:
exercise_df.head()

,Title,Type,BodyPart,Equipment,Level
0,Partner plank band row,Strength,Abdominals,Bands,Intermediate
1,Banded crunch isometric hold,Strength,Abdominals,Bands,Intermediate
2,FYR Banded Plank Jack,Strength,Abdominals,Bands,Intermediate
3,Banded crunch,Strength,Abdominals,Bands,Intermediate
4,Crunch,Strength,Abdominals,Bands,Intermediate


In [101]:
exercise_df.isnull().sum()

Title         0
Type          0
BodyPart      0
Equipment    32
Level         0
dtype: int64

In [102]:
exercise_df = exercise_df.fillna("")

In [103]:
exercise_df["Type"] = exercise_df["Type"].str.lower()

exercise_df["BodyPart"] = exercise_df["BodyPart"].str.lower()

exercise_df["Equipment"] = exercise_df["Equipment"].str.lower()

exercise_df["Level"] = exercise_df["Level"].str.lower()

In [104]:
exercise_df.head()

,Title,Type,BodyPart,Equipment,Level
0,Partner plank band row,strength,abdominals,bands,intermediate
1,Banded crunch isometric hold,strength,abdominals,bands,intermediate
2,FYR Banded Plank Jack,strength,abdominals,bands,intermediate
3,Banded crunch,strength,abdominals,bands,intermediate
4,Crunch,strength,abdominals,bands,intermediate


### Feature Engineering

In [105]:
exercise_df["combined_features"] = (
    exercise_df["Type"] + " " +
    exercise_df["BodyPart"] + " " +
    exercise_df["Equipment"] + " " +
    exercise_df["Level"]
)

In [106]:
exercise_df[
    ["Title", "combined_features"]
].head()

,Title,combined_features
0,Partner plank band row,strength abdominals bands intermediate
1,Banded crunch isometric hold,strength abdominals bands intermediate
2,FYR Banded Plank Jack,strength abdominals bands intermediate
3,Banded crunch,strength abdominals bands intermediate
4,Crunch,strength abdominals bands intermediate


### TF-IDF Vectorization

In [107]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [108]:
tfidf = TfidfVectorizer()

In [109]:
tfidf_matrix = tfidf.fit_transform(
    exercise_df["combined_features"]
)

In [110]:
tfidf_matrix.shape

(2918, 45)

### Cosine Similarity

In [111]:
from sklearn.metrics.pairwise import cosine_similarity

In [112]:
cosine_sim = cosine_similarity(tfidf_matrix)

In [113]:
cosine_sim.shape

(2918, 2918)

### Reccomendation Function

In [114]:
def recommend_workout(user_input, top_n=5):

    user_vector = tfidf.transform([user_input])

    similarity = cosine_similarity(
        user_vector,
        tfidf_matrix
    )

    similarity_scores = similarity.flatten()

    sorted_indices = similarity_scores.argsort()[::-1]

    recommendations = []

    added_titles = set()

    for idx in sorted_indices:

        title = exercise_df.iloc[idx]["Title"]

        if title not in added_titles:

            added_titles.add(title)

            workout = exercise_df.iloc[idx].copy()

            workout["Similarity Score"] = round(
                similarity_scores[idx],
                3
            )

            recommendations.append(workout)

        if len(recommendations) == top_n:
            break

    return pd.DataFrame(recommendations)[
        [
            "Title",
            "Type",
            "BodyPart",
            "Equipment",
            "Level",
            "Similarity Score"
        ]
    ]

In [115]:
recommend_workout(
    "strength chest beginner dumbbell"
)

,Title,Type,BodyPart,Equipment,Level,Similarity Score
1114,Hammer Grip Incline DB Bench Press,strength,chest,dumbbell,beginner,1.0
1121,One-Arm Flat Bench Dumbbell Flye,strength,chest,dumbbell,beginner,1.0
1109,Reverse-grip incline dumbbell bench press,strength,chest,dumbbell,beginner,1.0
1117,Incline Dumbbell Flyes - With A Twist,strength,chest,dumbbell,beginner,1.0
1116,Incline Dumbbell Bench With Palms Facing In,strength,chest,dumbbell,beginner,1.0


### Additional Testing

In [116]:
recommend_workout(
    "cardio legs beginner body only"
)

,Title,Type,BodyPart,Equipment,Level,Similarity Score
2169,Vertical Mountain Climber,cardio,quadriceps,body only,beginner,0.937
2167,Defensive Slide,cardio,quadriceps,body only,beginner,0.937
2165,Fast Kick With Arm Circles,cardio,quadriceps,body only,beginner,0.937
2171,Football Up-Down,cardio,quadriceps,body only,beginner,0.937
2175,Slow Jog,cardio,quadriceps,body only,beginner,0.937


In [117]:
recommend_workout(
    "strength back intermediate barbell"
)

,Title,Type,BodyPart,Equipment,Level,Similarity Score
1659,30 Back Underhand Bent-Over Barbell Row,strength,middle back,barbell,intermediate,0.776
1652,Barbell bent-over row,strength,middle back,barbell,intermediate,0.776
1639,One-Arm Long Bar Row,strength,middle back,barbell,intermediate,0.776
1640,Bent Over Barbell Row,strength,middle back,barbell,intermediate,0.776
1642,Pendlay Row,strength,middle back,barbell,intermediate,0.776


### Fitness Dataset

In [118]:
fitness_df = pd.read_csv("datasets/fitness.csv")

In [119]:
fitness_df.head()

,ID,Exercise,Calories Burn,Dream Weight,Actual Weight,Age,Gender,Duration,Heart Rate,BMI,Weather Conditions,Exercise Intensity
0,1,Exercise 2,286.959851,91.892531,96.301115,45,Male,37,170,29.426275,Rainy,5
1,2,Exercise 7,343.453036,64.165097,61.104668,25,Male,43,142,21.286346,Rainy,5
2,3,Exercise 4,261.223465,70.846224,71.766724,20,Male,20,148,27.899592,Cloudy,4
3,4,Exercise 5,127.183858,79.477008,82.984456,33,Male,39,170,33.729552,Sunny,10
4,5,Exercise 10,416.318374,89.960226,85.643174,29,Female,34,118,23.286113,Cloudy,3


In [120]:
fitness_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3864 entries, 0 to 3863
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  3864 non-null   int64  
 1   Exercise            3864 non-null   object 
 2   Calories Burn       3864 non-null   float64
 3   Dream Weight        3864 non-null   float64
 4   Actual Weight       3864 non-null   float64
 5   Age                 3864 non-null   int64  
 6   Gender              3864 non-null   object 
 7   Duration            3864 non-null   int64  
 8   Heart Rate          3864 non-null   int64  
 9   BMI                 3864 non-null   float64
 10  Weather Conditions  3864 non-null   object 
 11  Exercise Intensity  3864 non-null   int64  
dtypes: float64(4), int64(5), object(3)
memory usage: 362.4+ KB


In [121]:
fitness_df.columns

Index(['ID', 'Exercise', 'Calories Burn', 'Dream Weight', 'Actual Weight',
       'Age', 'Gender', 'Duration', 'Heart Rate', 'BMI', 'Weather Conditions',
       'Exercise Intensity'],
      dtype='object')

In [122]:
fitness_df.isnull().sum()

ID                    0
Exercise              0
Calories Burn         0
Dream Weight          0
Actual Weight         0
Age                   0
Gender                0
Duration              0
Heart Rate            0
BMI                   0
Weather Conditions    0
Exercise Intensity    0
dtype: int64

### Data Cleaning

In [123]:
fitness_df = fitness_df.fillna("")

In [124]:
fitness_df["Gender"] = (
    fitness_df["Gender"].str.lower()
)

fitness_df["Weather Conditions"] = (
    fitness_df["Weather Conditions"].str.lower()
)

fitness_df["Exercise"] = (
    fitness_df["Exercise"].str.lower()
)

In [125]:
fitness_df.head()

,ID,Exercise,Calories Burn,Dream Weight,Actual Weight,Age,Gender,Duration,Heart Rate,BMI,Weather Conditions,Exercise Intensity
0,1,exercise 2,286.959851,91.892531,96.301115,45,male,37,170,29.426275,rainy,5
1,2,exercise 7,343.453036,64.165097,61.104668,25,male,43,142,21.286346,rainy,5
2,3,exercise 4,261.223465,70.846224,71.766724,20,male,20,148,27.899592,cloudy,4
3,4,exercise 5,127.183858,79.477008,82.984456,33,male,39,170,33.729552,sunny,10
4,5,exercise 10,416.318374,89.960226,85.643174,29,female,34,118,23.286113,cloudy,3


### User Profile Generator

In [126]:
fitness_df["Exercise Intensity"].unique()

array([ 5,  4, 10,  3,  2,  1,  6,  9,  7,  8], dtype=int64)

In [127]:
def create_user_profile(bmi, intensity):

    if bmi > 25:
        workout_type = "cardio"
    else:
        workout_type = "strength"

    if intensity <= 3:
        level = "beginner"

    elif intensity <= 7:
        level = "intermediate"

    else:
        level = "expert"

    body_part = "legs"
    equipment = "body only"

    profile = (
        workout_type + " " +
        body_part + " " +
        equipment + " " +
        level
    )

    return profile

In [128]:
create_user_profile(
    bmi=30,
    intensity=2
)

'cardio legs body only beginner'

### Connect to Reccomendation System

In [129]:
user_profile = create_user_profile(
    bmi=30,
    intensity=2
)

recommend_workout(user_profile)

,Title,Type,BodyPart,Equipment,Level,Similarity Score
2169,Vertical Mountain Climber,cardio,quadriceps,body only,beginner,0.937
2167,Defensive Slide,cardio,quadriceps,body only,beginner,0.937
2165,Fast Kick With Arm Circles,cardio,quadriceps,body only,beginner,0.937
2171,Football Up-Down,cardio,quadriceps,body only,beginner,0.937
2175,Slow Jog,cardio,quadriceps,body only,beginner,0.937
